In [1]:
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 bitsandbytes==0.43.3 trl==0.10.1 datasets

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 94.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 77.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [3]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

print("Current Directory:", os.getcwd())

fatal: destination path 'Active-Reading--Pattern-Recognition' already exists and is not an empty directory.
/content/Active-Reading--Pattern-Recognition
Current Directory: /content/Active-Reading--Pattern-Recognition


In [1]:
# Connect to hugging face (Add a token with name "HF_TOKEN" from Hugging Face into Secrets here in Colab)
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face.")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it to access gated models.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

Successfully logged into Hugging Face.


In [4]:
!pip install --upgrade transformers tokenizers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 39.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.34.2
    Uninstalling accelerate-0.34.2:
      Successfully uninstalled accelerate-0.34.2


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

# LoRA
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()

model.print_trainable_parameters()

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


FINE-TUNING FOR **REPETITION** DIRECTLY FROM ORIGINAL DATASET

In [4]:
import json
from datasets import Dataset

INPUT_CORPUS = "Datasets/simple_wiki_corpus.json"
with open(INPUT_CORPUS, "r", encoding="utf-8") as f:
    corpus = json.load(f)

repetition_data = []

for entry in corpus:
    text = entry["text"].strip()

    chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 10]

    for c in chunks:
        repetition_data.append({
            "instruction": f"Repeat the following text exactly as provided to ensure factual retention for {entry['doc_name']}:",
            "input": c,
            "output": c  # Repetition Task so Input == Output
        })

dataset_repetition = Dataset.from_list(repetition_data)
print(f"Total samples for training: {len(dataset_repetition)}")

Total samples for training: 96067


In [5]:
from datasets import load_dataset, Dataset, concatenate_datasets

# Qwen ChatML format
def format_repetition_qwen(example):
    text = (
        f"<|im_start|>system\nYou are a verbatim memory assistant. Your goal is to repeat the input text exactly.<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}\n\n{example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}

formatted_dataset_repetition = dataset_repetition.map(format_repetition_qwen, remove_columns=dataset_repetition.column_names)

# DCLM (10% mixing)
dataset_dclm = load_dataset("mlfoundations/dclm-baseline-1.0", split="train", streaming=True)

num_repetition = len(formatted_dataset_repetition)
num_dclm_needed = max(1, num_repetition // 9)

print(f"Repetition samples: {num_repetition}")
print(f"DCLM samples needed: {num_dclm_needed}")

dclm_samples = []
for i, example in enumerate(dataset_dclm.take(num_dclm_needed)):
    formatted_text = (
        f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
        f"<|im_start|>user\nContinue the following text: {example['text'][:200]}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['text'][:500]}<|im_end|>"
    )
    dclm_samples.append({"text": formatted_text})

dataset_dclm_final = Dataset.from_list(dclm_samples)

# Mixing & Shuffling
final_dataset = concatenate_datasets([formatted_dataset_repetition, dataset_dclm_final])
final_dataset = final_dataset.shuffle(seed=42)

print(f"Final combined dataset size: {len(final_dataset)}")

Map:   0%|          | 0/96067 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/27838 [00:00<?, ?it/s]

Repetition samples: 96067
DCLM samples needed: 10674
Final combined dataset size: 106741


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen_repetition_checkpoints",
    max_seq_length=1024,
    dataset_text_field="text",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    logging_steps=5,
    fp16=True,
    seed=42,
    gradient_checkpointing=True,
    report_to="none"
)

model.enable_input_require_grads()
trainer = SFTTrainer(
    model=model,
    train_dataset=final_dataset,
    args=sft_config,
)

trainer.train(resume_from_checkpoint=False) # Change this to True after first run

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/106741 [00:00<?, ? examples/s]

In [ ]:
def test_repetition(input_text):
    prompt = (
        f"<|im_start|>system\nYou are a verbatim memory assistant. Your goal is to repeat the input text exactly.<|im_end|>\n"
        f"<|im_start|>user\nRepeat the following text exactly: {input_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "assistant" in full_output:
        return full_output.split("assistant")[-1].strip()
    return full_output